# PCA and Sample QC Analysis

This notebook performs sample-level QC using PCA:
- PCA visualization with sample grouping
- Variance explained
- Sample correlation heatmap
- Outlier detection

**Input**: Count matrix (genes x samples) or normalized expression
**Outputs**: PCA plots, correlation heatmap

In [ ]:
# Parameters
N_TOP_GENES = 500  # Top variable genes for PCA
GROUP_COLUMN = None  # Column name for sample grouping (or None)
LOG_TRANSFORM = True  # Apply log2 transform
FIGURE_DPI = 150

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import linkage, dendrogram
from galaxy_notebook import get, put, list_inputs

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = FIGURE_DPI

In [ ]:
# List inputs
print("Available inputs:")
for inp in list_inputs():
    print(f"  [{inp['index']}] {inp['name']}")

In [ ]:
# Load count matrix
counts = get(0)

# Identify gene column
gene_cols = ['gene', 'gene_id', 'Gene', 'GeneID', 'Geneid']
gene_col = None
for col in gene_cols:
    if col in counts.columns:
        gene_col = col
        break

if gene_col:
    counts = counts.set_index(gene_col)

# Get numeric columns only (samples)
sample_cols = counts.select_dtypes(include=[np.number]).columns.tolist()
counts = counts[sample_cols]

print(f"Loaded: {counts.shape[0]} genes x {counts.shape[1]} samples")
print(f"Samples: {sample_cols}")

In [ ]:
# Preprocessing
# Remove genes with zero counts
counts = counts.loc[counts.sum(axis=1) > 0]

# Log transform if requested
if LOG_TRANSFORM:
    counts_log = np.log2(counts + 1)
    print("Applied log2(x+1) transformation")
else:
    counts_log = counts

# Select top variable genes
gene_var = counts_log.var(axis=1).sort_values(ascending=False)
top_genes = gene_var.head(N_TOP_GENES).index
counts_subset = counts_log.loc[top_genes]

print(f"Selected top {N_TOP_GENES} variable genes")

## 1. PCA Analysis

In [ ]:
# Run PCA
X = counts_subset.T.values  # samples x genes
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=min(10, len(sample_cols)))
pca_coords = pca.fit_transform(X_scaled)

# Create PCA DataFrame
pca_df = pd.DataFrame(
    pca_coords[:, :4],
    index=sample_cols,
    columns=['PC1', 'PC2', 'PC3', 'PC4']
)

print(f"Variance explained:")
for i, var in enumerate(pca.explained_variance_ratio_[:5]):
    print(f"  PC{i+1}: {var*100:.1f}%")

In [ ]:
# PCA scatter plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# PC1 vs PC2
ax = axes[0]
scatter = ax.scatter(pca_df['PC1'], pca_df['PC2'], s=100, c='#3498db', edgecolor='white')

# Label points
for idx, row in pca_df.iterrows():
    ax.annotate(idx, (row['PC1'], row['PC2']), fontsize=8, alpha=0.8,
                xytext=(5, 5), textcoords='offset points')

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
ax.set_title('PCA: PC1 vs PC2')
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.axvline(0, color='gray', linestyle='--', alpha=0.5)

# Variance explained bar plot
ax = axes[1]
var_exp = pca.explained_variance_ratio_[:10] * 100
ax.bar(range(1, len(var_exp)+1), var_exp, color='#3498db', edgecolor='white')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Variance Explained (%)')
ax.set_title('Scree Plot')
ax.set_xticks(range(1, len(var_exp)+1))

# Add cumulative line
ax2 = ax.twinx()
ax2.plot(range(1, len(var_exp)+1), np.cumsum(var_exp), 'r-o', markersize=5)
ax2.set_ylabel('Cumulative Variance (%)', color='red')
ax2.tick_params(axis='y', labelcolor='red')

plt.tight_layout()
put(fig, 'pca_analysis.png')
plt.show()

## 2. Sample Correlation Heatmap

In [ ]:
# Calculate sample correlation
corr_matrix = counts_log.corr()

# Hierarchical clustering
fig, ax = plt.subplots(figsize=(max(10, len(sample_cols)*0.5), max(8, len(sample_cols)*0.4)))

# Cluster samples
linkage_matrix = linkage(corr_matrix, method='average')

# Plot heatmap with clustering
g = sns.clustermap(corr_matrix, 
                   cmap='RdBu_r',
                   vmin=0.8, vmax=1.0,
                   annot=True if len(sample_cols) <= 10 else False,
                   fmt='.2f',
                   figsize=(max(10, len(sample_cols)*0.5), max(8, len(sample_cols)*0.4)))
g.fig.suptitle('Sample Correlation Heatmap', y=1.02)

put(g.fig, 'sample_correlation.png')
plt.show()

## 3. Sample Statistics

In [ ]:
# Calculate sample statistics
sample_stats = pd.DataFrame({
    'total_counts': counts.sum(),
    'detected_genes': (counts > 0).sum(),
    'mean_correlation': corr_matrix.mean() - 1/(len(sample_cols)),  # Exclude self-correlation
})

# Flag potential outliers (low correlation with other samples)
corr_threshold = sample_stats['mean_correlation'].mean() - 2 * sample_stats['mean_correlation'].std()
sample_stats['potential_outlier'] = sample_stats['mean_correlation'] < corr_threshold

print("Sample Statistics:")
sample_stats

In [ ]:
# Plot sample statistics
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Total counts
sample_stats['total_counts'].plot(kind='bar', ax=axes[0], color='#3498db')
axes[0].set_title('Total Counts per Sample')
axes[0].set_ylabel('Total Counts')
axes[0].tick_params(axis='x', rotation=45)

# Detected genes
sample_stats['detected_genes'].plot(kind='bar', ax=axes[1], color='#2ecc71')
axes[1].set_title('Detected Genes per Sample')
axes[1].set_ylabel('Number of Genes')
axes[1].tick_params(axis='x', rotation=45)

# Mean correlation
colors = ['#e74c3c' if x else '#3498db' for x in sample_stats['potential_outlier']]
sample_stats['mean_correlation'].plot(kind='bar', ax=axes[2], color=colors)
axes[2].axhline(corr_threshold, color='red', linestyle='--', label='Outlier threshold')
axes[2].set_title('Mean Sample Correlation')
axes[2].set_ylabel('Correlation')
axes[2].tick_params(axis='x', rotation=45)
axes[2].legend()

plt.tight_layout()
put(fig, 'sample_statistics.png')
plt.show()

In [ ]:
# Summary
outliers = sample_stats[sample_stats['potential_outlier']].index.tolist()

summary = f"""
# PCA and Sample QC Summary

## Dataset
- Genes: {counts.shape[0]:,}
- Samples: {counts.shape[1]}
- Top variable genes used: {N_TOP_GENES}

## PCA Results
- PC1 variance explained: {pca.explained_variance_ratio_[0]*100:.1f}%
- PC2 variance explained: {pca.explained_variance_ratio_[1]*100:.1f}%
- Cumulative (PC1-4): {sum(pca.explained_variance_ratio_[:4])*100:.1f}%

## Sample QC
- Mean total counts: {sample_stats['total_counts'].mean():,.0f}
- Mean detected genes: {sample_stats['detected_genes'].mean():,.0f}
- Potential outliers: {len(outliers)} ({', '.join(outliers) if outliers else 'None'})
"""

print(summary)
put(summary, 'pca_summary.md')
put(sample_stats.reset_index(), 'sample_statistics.tsv')
put(pca_df.reset_index(), 'pca_coordinates.tsv')

In [ ]:
print("\n=== Analysis Complete ===")
print("Generated outputs:")
print("  - pca_analysis.png")
print("  - sample_correlation.png")
print("  - sample_statistics.png")
print("  - pca_summary.md")
print("  - sample_statistics.tsv")
print("  - pca_coordinates.tsv")